# Stress Prediction — 거리 적응형(Adaptive) 블렌드

EDA에서 확인된 핵심 사실: test의 12.8%(383/3000)가 train의 어떤 샘플과 사실상 동일하고,
일부는 완전히 동일합니다(distance=0). 라벨 노이즈도 없습니다(같은 입력=같은 타겟).

**아이디어**: 지금 v34는 모든 샘플에 고정된 0.28 비중으로 pair를 섞습니다.
대신 **최근접 거리가 가까운 샘플일수록 pair(그 이웃의 실제 타겟값)를 더 신뢰**하고,
먼 샘플일수록 tree를 더 신뢰하도록 **샘플별로 가중치를 다르게** 줍니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from scipy.spatial import cKDTree

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.54

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


## 거리 기반 최근접이웃 + 적응형 가중치

`nearest_dist`는 EDA와 동일한 방식(수치형 컬럼 표준화 + 유클리드 거리)으로 계산합니다.
`adaptive_weight(dist)`는 거리 0일 때 최대 신뢰(0.95), 멀어질수록 원래 v34 비중(0.28)으로 수렴합니다.


In [2]:

def nearest_neighbor_lookup(train_raw, val_raw, target, numeric_cols):
    train_numeric = train_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    val_numeric = val_raw[numeric_cols].fillna(train_raw[numeric_cols].median())

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    val_scaled = scaler.transform(val_numeric)

    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(val_scaled, k=1)
    nearest_target = target.to_numpy(dtype=float)[nearest_idx]
    return nearest_dist, nearest_target


def adaptive_weight(distance, max_weight=0.95, min_weight=0.28, scale=0.5):
    """거리 0 -> max_weight, 거리가 커질수록 min_weight로 수렴 (지수 감쇠)."""
    decay = np.exp(-distance / scale)
    return min_weight + (max_weight - min_weight) * decay


## 5-Fold CV — 적응형 블렌드 vs v34 고정 블렌드 비교

In [3]:

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

raw_X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
raw_X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

numeric_cols = train.drop(columns=[TARGET, ID_COLUMN]).select_dtypes(include=["number"]).columns.tolist()
print("최근접이웃 탐색에 쓰는 수치형 컬럼:", numeric_cols)

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
oof_adaptive = np.zeros(len(y))
oof_reference = np.zeros(len(y))  # v34 고정 0.72/0.28 블렌드 (sanity check)
fold_rows = []

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
    val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].to_numpy(dtype=float)

    pred_tree = tree_prediction(train_feat, val_feat, y_train)

    train_raw_fold = train.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = train.iloc[val_idx].reset_index(drop=True)
    nearest_dist, nearest_target = nearest_neighbor_lookup(train_raw_fold, val_raw_fold, y_train, numeric_cols)

    w = adaptive_weight(nearest_dist)
    adaptive_pred = np.clip(np.round(w * nearest_target + (1 - w) * pred_tree, 2), 0.0, 1.0)
    reference_pred = np.clip(np.round(0.28 * nearest_target + 0.72 * pred_tree, 2), 0.0, 1.0)

    oof_adaptive[val_idx] = adaptive_pred
    oof_reference[val_idx] = reference_pred

    fold_rows.append({
        "Fold": fold,
        "Adaptive_MAE": mean_absolute_error(y_val, adaptive_pred),
        "Fixed028_MAE": mean_absolute_error(y_val, reference_pred),
        "mean_weight": w.mean(),
    })
    print(f"Fold {fold} done")

fold_df = pd.DataFrame(fold_rows)
display(fold_df)
print()
print("### 적응형 블렌드 CV MAE:", f"{mean_absolute_error(y, oof_adaptive):.6f}")
print("### 고정 0.28 블렌드(1-NN 버전) CV MAE:", f"{mean_absolute_error(y, oof_reference):.6f}")
print("(참고) v34 원본(28개 조합 pair) CV MAE: 0.148033")


최근접이웃 탐색에 쓰는 수치형 컬럼: ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density', 'mean_working']
Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done


,Fold,Adaptive_MAE,Fixed028_MAE,mean_weight
0,1,0.153450,0.148333,0.465917
1,2,0.164650,0.160650,0.454182
2,3,0.167983,0.162900,0.456530
3,4,0.173700,0.169033,0.456750
4,5,0.167233,0.163883,0.459606



### 적응형 블렌드 CV MAE: 0.165403
### 고정 0.28 블렌드(1-NN 버전) CV MAE: 0.160960
(참고) v34 원본(28개 조합 pair) CV MAE: 0.148033


## adaptive_weight 파라미터 그리드서치 (선택, 시간 되면)

`max_weight`, `scale`을 몇 가지 바꿔가며 더 낮은 CV MAE가 나오는 조합을 찾습니다.
거리/타겟 계산은 위에서 이미 했으니, 가중치 함수만 바꿔서 빠르게 재평가합니다.


In [4]:

# 위 CV 루프에서 fold별 (nearest_dist, nearest_target, pred_tree, y_val)을 다시 계산해 재사용
fold_cache = []
for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
    val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].to_numpy(dtype=float)
    pred_tree = tree_prediction(train_feat, val_feat, y_train)
    train_raw_fold = train.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = train.iloc[val_idx].reset_index(drop=True)
    nearest_dist, nearest_target = nearest_neighbor_lookup(train_raw_fold, val_raw_fold, y_train, numeric_cols)
    fold_cache.append((val_idx, nearest_dist, nearest_target, pred_tree, y_val))
    print(f"Fold {fold} cached")

grid_rows = []
for max_weight in [0.85, 0.90, 0.95, 0.99]:
    for scale in [0.2, 0.3, 0.5, 0.8]:
        oof = np.zeros(len(y))
        for val_idx, nearest_dist, nearest_target, pred_tree, y_val in fold_cache:
            w = adaptive_weight(nearest_dist, max_weight=max_weight, min_weight=0.28, scale=scale)
            oof[val_idx] = np.clip(np.round(w * nearest_target + (1 - w) * pred_tree, 2), 0.0, 1.0)
        mae = mean_absolute_error(y, oof)
        grid_rows.append({"max_weight": max_weight, "scale": scale, "cv_mae": mae})

grid_df = pd.DataFrame(grid_rows).sort_values("cv_mae").reset_index(drop=True)
display(grid_df.head(10))
print("(참고) v34 원본 CV MAE: 0.148033")


Fold 1 cached
Fold 2 cached
Fold 3 cached
Fold 4 cached
Fold 5 cached


,max_weight,scale,cv_mae
0,0.85,0.2,0.161117
1,0.95,0.2,0.161123
2,0.90,0.2,0.161127
3,0.99,0.2,0.161127
4,0.85,0.3,0.161833
5,0.90,0.3,0.161933
6,0.95,0.3,0.162003
7,0.99,0.3,0.162043
8,0.85,0.5,0.164737
9,0.90,0.5,0.165070


(참고) v34 원본 CV MAE: 0.148033


## 최종 제출 파일 생성

CV MAE가 0.148033보다 낮으면 아래 셀의 `BEST_MAX_WEIGHT`, `BEST_SCALE`을 grid 결과 1위로 맞추고 실행하세요.


In [5]:

BEST_MAX_WEIGHT = 0.95  # grid 결과에 맞게 수정
BEST_SCALE = 0.5        # grid 결과에 맞게 수정

final_tree = tree_prediction(raw_X, raw_X_test, y)
final_nearest_dist, final_nearest_target = nearest_neighbor_lookup(train, test, y, numeric_cols)
final_w = adaptive_weight(final_nearest_dist, max_weight=BEST_MAX_WEIGHT, min_weight=0.28, scale=BEST_SCALE)
final_prediction = np.clip(np.round(final_w * final_nearest_target + (1 - final_w) * final_tree, 2), 0.0, 1.0)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_adaptive_blend_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
print("평균 적용 가중치:", final_w.mean())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_adaptive_blend_0806.csv
Prediction range: 0.0 1.0
평균 적용 가중치: 0.47181536661674256


,ID,stress_score
0,TEST_0000,0.68
1,TEST_0001,0.97
2,TEST_0002,0.30
3,TEST_0003,0.35
4,TEST_0004,0.37
5,TEST_0005,0.44
6,TEST_0006,0.78
7,TEST_0007,0.45
8,TEST_0008,0.76
9,TEST_0009,0.88
